### ЗАДАЧА: Пакетная загрузка отгрузок (try/except + custom exceptions)

Из внешней логистической системы приходят строки с отгрузками.
Нужно безопасно распарсить данные, отделить валидные записи от ошибок
и посчитать несколько итоговых метрик.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `ShipmentError`
   - `RowFormatError`
   - `WeightError`
   - `PriorityError`
   - `RegionError`.

2. Функцию `parse_shipment(row)`:
   - формат строки: `shipment_id,client,weight,priority,region`
   - `weight` должен быть числом и `> 0`
   - допустимые приоритеты: `standard`, `express`, `vip`
   - допустимые регионы: `RU`, `KZ`, `BY`
   - при ошибке конвертации веса использовать `raise ... from ...`.

3. Функцию `load_shipments(rows)`:
   - вернуть `(shipments, errors)`
   - ошибки хранить как `(row, error_type, message)`
   - не останавливать цикл на первой ошибке.

4. Вывести:
   - число валидных отгрузок,
   - ошибки по типам,
   - суммарный вес только для `express` и `vip`,
   - клиента-лидера по суммарному весу среди валидных записей.

In [ ]:
rows = [
    'S-100,Acme,12.5,express,RU',
    'S-101,Beta,0,standard,RU',
    'S-102,Acme,abc,vip,KZ',
    'S-103,Delta,8.5,urgent,BY',
    'S-104,Gamma,15,vip,UZ',
    'S-105,Acme,4.0,standard,KZ',
    'S-106,Beta,9.5,express,BY',
]


class ShipmentError(Exception):
    pass


class RowFormatError(ShipmentError):
    pass


class WeightError(ShipmentError):
    pass


class PriorityError(ShipmentError):
    pass


class RegionError(ShipmentError):
    pass


def parse_shipment(row):
    # TODO: распарсить строку и провалидировать weight, priority, region
    parts = row.split(",")
    if len(parts) != 5:
        raise RowFormatError("Неверный формат строки")
    
    shipment_id, client, weight, priority, region = parts

    # TODO: при ошибке конвертации weight использовать raise ... from ...
    try:
        weight = float(weight)
        if weight <= 0:
            raise WeightError("Вес должен быть положительным")
    except ValueError as e:
        raise WeightError("Некоректное значение веса") from e
    
    if priority not in ["express", "vip"]:
        raise PriorityError("Неверный приоритет")
    if region not in ["RU", "KZ", "BY", "UZ"]:
        raise RegionError("Неверный регион")
    
    return {
        "shipment_id": shipment_id,
        "client": client,
        "weight": weight,
        "priority": priority,
        "region": region
    }

    


def load_shipments(rows):
    # TODO: вернуть (shipments, errors)
    shipments = []
    errors = []

    for row in rows:
        try: 
            shipment = parse_shipment(row)
            shipments.append(shipment)
        except ShipmentError as e:
            errors.append((row, str(e)))
    return shipments, errors


# TODO: вызвать load_shipments(rows)
shipments, errors = load_shipments(rows)

# TODO: вывести число валидных отгрузок и число ошибок
print("Число валидных отгрузок: ", len(shipments))
print("Количество ошибок: ", len(errors))

# TODO: вывести ошибки по типам
from collections import Counter
error_types_counter = Counter(type(e[1]) for e in errors)

for error_type, count in error_types_counter.items():
    print(f"{error_type.__name__}: {count}")

# TODO: посчитать premium_weight только для express/vip
premium_weight = sum(s["weight"] for s in shipments if s["priority"] in ["express", "vip"])
print("Общий вес: ", premium_weight)

# TODO: найти клиента-лидера по суммарному весу
from collections import defaultdict
client_weights = defaultdict(float)

for s in shipments:
    client_weights[s["client"]] += s["weight"]
leader_client = max(client_weights, key=client_weights.get)
print("Клиент-лидер по весу: ", leader_client, "с весом ", client_weights[leader_client])